# KPI Calculation


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration
PROCESSED_DIR = Path('../processed')
YEAR = 2022

# Ensure output directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Columns required for KPI calculation
# Added 'qa_flags' for filtering and 'avg_mph' for speed stats
REQUIRED_COLS = [
    'tpep_pickup_datetime',
    'total_amount',
    'trip_distance',
    'trip_duration_minutes',
    'passenger_count',
    'fare_amount',
    'tip_amount',
    'qa_flags',
    'avg_mph'
]


In [2]:
# List all processed files for the year
files = sorted(PROCESSED_DIR.glob(f'yellow_tripdata_{YEAR}-*.parquet'))
print(f"Found {len(files)} files to process.")


Found 12 files to process.


In [3]:
def get_percentiles(series, percentiles=[0.5, 0.95]):
    """
    Calculates percentiles for a series.
    Returns a Series with index like 'p50', 'p95'.
    """
    results = {}
    # Handle empty or all-NaN series
    if series.empty or series.isna().all():
        for p in percentiles:
            results[f'p{int(p*100)}'] = np.nan
        return pd.Series(results)
        
    try:
        values = series.quantile(percentiles)
        # Convert to dict to avoid float indexing issues
        v_dict = values.to_dict()
        
        for p in percentiles:
            key = f'p{int(round(p*100))}'
            # Try to get by exact float, or close enough?
            # Usually to_dict preserves the float key.
            if p in v_dict:
                results[key] = v_dict[p]
            else:
                # Fallback: try to find closest key? 
                # Or just assume it failed.
                results[key] = np.nan
                
    except Exception as e:
        # Fallback
        for p in percentiles:
            key = f'p{int(round(p*100))}'
            results[key] = np.nan
            
    return pd.Series(results)

def aggregate_chunk(df, frequency):
    """
    Resamples and aggregates a dataframe chunk.
    Returns a dataframe with sums, counts, and percentiles.
    """
    # Filter out flagged rows
    clean_df = df[df['qa_flags'] == 0].copy()
    
    if clean_df.empty:
        return pd.DataFrame()

    # Ensure datetime index
    temp_df = clean_df.set_index('tpep_pickup_datetime')
    
    # Add a trip count column
    temp_df['trip_count'] = 1
    
    # 1. Calculate Sums and Counts (Aggregatable)
    aggs = {
        'total_amount': 'sum',
        'trip_distance': 'sum',
        'trip_duration_minutes': 'sum',
        'passenger_count': 'sum',
        'tip_amount': 'sum',
        'fare_amount': 'sum',
        'trip_count': 'sum'
    }
    
    agg_df = temp_df.resample(frequency).agg(aggs)
    
    # 2. Calculate Percentiles
    # Group by frequency
    grouper = temp_df.groupby(pd.Grouper(freq=frequency))
    
    # Calculate percentiles for Duration and Speed
    duration_stats = grouper['trip_duration_minutes'].apply(lambda x: get_percentiles(x, [0.5, 0.95]))
    
    if isinstance(duration_stats, pd.Series):
        duration_df = duration_stats.apply(pd.Series)
    else:
        duration_df = duration_stats
        
    # Rename columns
    rename_map = {'p50': 'duration_p50', 'p95': 'duration_p95'}
    duration_df = duration_df.rename(columns=rename_map)
    
    # Fallback if rename didn't work
    if 'duration_p50' not in duration_df.columns:
        if len(duration_df.columns) == 2:
            duration_df.columns = ['duration_p50', 'duration_p95']
        elif len(duration_df.columns) == 1:
            duration_df.columns = ['duration_p50']
    
    speed_stats = grouper['avg_mph'].apply(lambda x: get_percentiles(x, [0.5]))
    
    if isinstance(speed_stats, pd.Series):
        speed_df = speed_stats.apply(pd.Series)
    else:
        speed_df = speed_stats
        
    speed_df = speed_df.rename(columns={'p50': 'speed_p50'})
    if 'speed_p50' not in speed_df.columns and len(speed_df.columns) == 1:
        speed_df.columns = ['speed_p50']
    
    # Merge percentiles into agg_df
    agg_df = agg_df.join(duration_df, rsuffix='_dur').join(speed_df, rsuffix='_spd')
    
    return agg_df

def calculate_final_metrics(agg_df, frequency_name):
    """
    Calculates derived metrics (averages, percentages) from aggregated sums.
    Also handles Index(100) and Indexdow.
    """
    kpi_df = agg_df.copy()
    
    # Rename columns for clarity
    kpi_df = kpi_df.rename(columns={
        'total_amount': 'revenue',
        'trip_distance': 'total_distance',
        'trip_duration_minutes': 'total_duration_minutes',
        'passenger_count': 'total_passengers',
        'tip_amount': 'total_tips',
        'fare_amount': 'total_fare_revenue',
        'trip_count': 'total_trips'
    })
    
    # Calculate derived KPIs
    kpi_df['avg_trip_price'] = kpi_df['revenue'] / kpi_df['total_trips']
    kpi_df['avg_distance'] = kpi_df['total_distance'] / kpi_df['total_trips']
    kpi_df['avg_duration_minutes'] = kpi_df['total_duration_minutes'] / kpi_df['total_trips']
    kpi_df['avg_tip'] = kpi_df['total_tips'] / kpi_df['total_trips']
    
    # Avg Speed (mph)
    kpi_df['avg_speed_mph'] = kpi_df.apply(
        lambda row: row['total_distance'] / (row['total_duration_minutes'] / 60) 
        if row['total_duration_minutes'] > 0 else 0, axis=1
    )
    
    # Avg Tip %
    kpi_df['avg_tip_pct'] = kpi_df.apply(
        lambda row: (row['total_tips'] / row['total_fare_revenue']) * 100 
        if row['total_fare_revenue'] > 0 else 0, axis=1
    )
    
    # Index(100) - Base 100 on the first row
    if not kpi_df.empty:
        base_revenue = kpi_df['revenue'].iloc[0]
        base_trips = kpi_df['total_trips'].iloc[0]
        
        kpi_df['revenue_index_100'] = (kpi_df['revenue'] / base_revenue) * 100 if base_revenue else 0
        kpi_df['trips_index_100'] = (kpi_df['total_trips'] / base_trips) * 100 if base_trips else 0
        
    # Indexdow (Day of Week Index)
    # Only applicable for Daily data
    if frequency_name == 'Daily':
        kpi_df['day_of_week'] = kpi_df.index.dayofweek
        
        # Calculate average revenue per day of week
        dow_means = kpi_df.groupby('day_of_week')['revenue'].mean()
        
        # Map back to the dataframe
        kpi_df['revenue_index_dow'] = kpi_df.apply(
            lambda row: (row['revenue'] / dow_means[row['day_of_week']]) * 100 
            if dow_means[row['day_of_week']] else 0, axis=1
        )
        
        # Cleanup helper column
        kpi_df = kpi_df.drop(columns=['day_of_week'])

    return kpi_df

print("ok")

ok


In [4]:
import gc

# Initialize containers for aggregated parts
daily_parts = []
weekly_parts = []
monthly_parts = []

print("Starting iterative processing...")

for f in files:
    print(f"Processing {f.name}...")
    
    # Load chunk
    df_chunk = pd.read_parquet(f, columns=REQUIRED_COLS)
    
    # Ensure datetime
    df_chunk['tpep_pickup_datetime'] = pd.to_datetime(df_chunk['tpep_pickup_datetime'])
    
    # Filter for the target year only
    df_chunk = df_chunk[df_chunk['tpep_pickup_datetime'].dt.year == YEAR]
    
    if df_chunk.empty:
        print(f"  No data for {YEAR} in {f.name}, skipping...")
        continue
        
    # Aggregate for each frequency
    # Daily
    daily_agg = aggregate_chunk(df_chunk, 'D')
    if not daily_agg.empty:
        daily_parts.append(daily_agg)
    
    # Weekly
    weekly_agg = aggregate_chunk(df_chunk, 'W')
    if not weekly_agg.empty:
        weekly_parts.append(weekly_agg)
    
    # Monthly
    monthly_agg = aggregate_chunk(df_chunk, 'ME')
    if not monthly_agg.empty:
        monthly_parts.append(monthly_agg)
    
    # Clean up memory
    del df_chunk
    del daily_agg
    del weekly_agg
    del monthly_agg
    gc.collect()

print("All files processed. Combining and calculating final KPIs...")

def finalize_and_save(parts, frequency_name, filename):
    if not parts:
        print(f"No data found for {frequency_name} KPIs.")
        return pd.DataFrame()
        
    print(f"Finalizing {frequency_name} KPIs...")
    # Concatenate all parts
    full_agg = pd.concat(parts)
    
    # Group by index (datetime)
    # For Sums: Sum them
    # For Percentiles: We take the weighted average based on trip_count? 
    # Or simpler: take the value from the row with the max trip_count (the "main" chunk for that period)
    # This handles the split week issue by prioritizing the chunk that has more data for that week.
    
    # Define aggregation for the final merge
    # Sum columns
    sum_cols = ['total_amount', 'trip_distance', 'trip_duration_minutes', 
                'passenger_count', 'tip_amount', 'fare_amount', 'trip_count']
    
    # Percentile columns (take the one from the largest chunk)
    # We can't easily do "arg max" in standard agg.
    # Workaround: Sort by trip_count descending, then take 'first'.
    
    full_agg = full_agg.sort_values('trip_count', ascending=False)
    
    agg_rules = {col: 'sum' for col in sum_cols}
    
    # Add rules for percentile columns if they exist
    for col in full_agg.columns:
        if col not in sum_cols and col != 'tpep_pickup_datetime':
            agg_rules[col] = 'first' # Takes the value from the largest chunk (due to sort)
            
    final_sums = full_agg.groupby(level=0).agg(agg_rules)
    
    # Sort index
    final_sums = final_sums.sort_index()
    
    # Calculate derived metrics
    final_kpi = calculate_final_metrics(final_sums, frequency_name)
    
    # Save
    output_path = PROCESSED_DIR / filename
    final_kpi.to_csv(output_path)
    print(f"Saved {frequency_name} KPIs to {output_path}")
    return final_kpi

# Finalize Daily
kpi_daily = finalize_and_save(daily_parts, "Daily", f'kpi_daily_{YEAR}.csv')

# Finalize Weekly
kpi_weekly = finalize_and_save(weekly_parts, "Weekly", f'kpi_weekly_{YEAR}.csv')

# Finalize Monthly
kpi_monthly = finalize_and_save(monthly_parts, "Monthly", f'kpi_monthly_{YEAR}.csv')

# Display sample
if not kpi_monthly.empty:
    print("\nMonthly KPI Sample:")
    print(kpi_monthly.head())


Starting iterative processing...
Processing yellow_tripdata_2022-01.parquet...
Processing yellow_tripdata_2022-02.parquet...
Processing yellow_tripdata_2022-03.parquet...
Processing yellow_tripdata_2022-04.parquet...
Processing yellow_tripdata_2022-05.parquet...
Processing yellow_tripdata_2022-06.parquet...
Processing yellow_tripdata_2022-07.parquet...
Processing yellow_tripdata_2022-08.parquet...
Processing yellow_tripdata_2022-09.parquet...
Processing yellow_tripdata_2022-10.parquet...
Processing yellow_tripdata_2022-11.parquet...
Processing yellow_tripdata_2022-12.parquet...
All files processed. Combining and calculating final KPIs...
Finalizing Daily KPIs...
Saved Daily KPIs to ../processed/kpi_daily_2022.csv
Finalizing Weekly KPIs...
Saved Weekly KPIs to ../processed/kpi_weekly_2022.csv
Finalizing Monthly KPIs...
Saved Monthly KPIs to ../processed/kpi_monthly_2022.csv

Monthly KPI Sample:
                           revenue  total_distance  total_duration_minutes  \
tpep_pickup_dat

# KPI Definitions

| KPI Name | Goal | Formula | Unit |
|----------|------|---------|------|
| **Revenue** | Track total income generated from trips. | $\sum(\text{total\_amount})$ | USD ($) |
| **Total Trips** | Monitor the volume of taxi usage. | $\text{count}(\text{rows})$ | Count |
| **Avg Trip Price** | Understand the average cost of a trip for a passenger. | $\frac{\sum(\text{total\_amount})}{\text{count}(\text{rows})}$ | USD ($) |
| **Avg Distance** | Track the average length of trips. | $\frac{\sum(\text{trip\_distance})}{\text{count}(\text{rows})}$ | Miles |
| **Avg Duration** | Monitor trip efficiency and traffic conditions. | $\frac{\sum(\text{trip\_duration\_minutes})}{\text{count}(\text{rows})}$ | Minutes |
| **Avg Speed** | Assess traffic flow and trip efficiency. | $\frac{\sum(\text{trip\_distance})}{\sum(\text{trip\_duration\_minutes}) / 60}$ | MPH |
| **Total Passengers** | Track the total number of people served. | $\sum(\text{passenger\_count})$ | Count |
| **Avg Tip** | Monitor driver gratuity earnings. | $\frac{\sum(\text{tip\_amount})}{\text{count}(\text{rows})}$ | USD ($) |
| **Avg Tip %** | Understand tipping behavior relative to fare. | $\frac{\sum(\text{tip\_amount})}{\sum(\text{fare\_amount})} \times 100$ | Percentage (%) |
| **Duration p50** | Median trip duration. | $\text{Median}(\text{trip\_duration\_minutes})$ | Minutes |
| **Duration p95** | 95th percentile of trip duration. | $\text{Percentile}(\text{trip\_duration\_minutes}, 95)$ | Minutes |
| **Speed p50** | Median trip speed. | $\text{Median}(\text{avg\_mph})$ | MPH |
| **Index(100)** | Relative performance to start of year. | $\frac{\text{Value}}{\text{Value}_{\text{start}}} \times 100$ | Index |
| **Index DOW** | Relative performance to day-of-week average. | $\frac{\text{Value}}{\text{Avg}_{\text{DOW}}} \times 100$ | Index |
